<a href="https://colab.research.google.com/github/Halidh-Ahamed/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [12]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [13]:
feature_df = con.sql(f"""
SELECT
    report_date,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    client_has_gsc,
    client_has_ga4,
    gsc_data_available
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [14]:
feature_df["avg_position"] = (
    feature_df["gsc_sum_position"] /
    feature_df["gsc_impressions"].replace(0, 1)
)

feature_df["ctr"] = (
    feature_df["gsc_clicks"] /
    feature_df["gsc_impressions"].replace(0, 1)
)

In [15]:
feature_df[[
    "gsc_impressions",
    "gsc_clicks",
    "avg_position",
    "ctr"
]].describe()

,gsc_impressions,gsc_clicks,avg_position,ctr
count,9.841378e+06,9.841378e+06,9.841378e+06,9.841378e+06
mean,2.851812e+01,8.350782e-02,5.807216e+00,1.130408e-03
std,1.559266e+02,7.814341e-01,1.424255e+01,1.828814e-02
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,6.000000e+00,0.000000e+00,4.666667e+00,0.000000e+00
max,4.008400e+04,2.740000e+02,4.980000e+02,1.000000e+00


In [16]:
feature_df[[
    "gsc_impressions",
    "gsc_clicks",
    "avg_position",
    "ctr"
]].quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

,gsc_impressions,gsc_clicks,avg_position,ctr
0.25,0.0,0.0,0.000000,0.000000
0.50,0.0,0.0,0.000000,0.000000
0.75,6.0,0.0,4.666667,0.000000
0.90,54.0,0.0,18.000000,0.000000
0.95,135.0,0.0,35.074074,0.000000
0.99,509.0,2.0,76.101211,0.020408


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [17]:
feature_df[["gsc_impressions", "gsc_clicks"]].corr()

,gsc_impressions,gsc_clicks
gsc_impressions,1.000000,0.596687
gsc_clicks,0.596687,1.000000


In [18]:
import pandas as pd

In [19]:
feature_df.groupby(
    pd.cut(
        feature_df["avg_position"],
        bins=[0, 5, 10, 20, 35, 100]
    )
)["ctr"].mean()

/tmp/ipykernel_826/2827260475.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  feature_df.groupby(


,ctr
avg_position,
"(0, 5]",0.004540
"(5, 10]",0.003083
"(10, 20]",0.002770
"(20, 35]",0.001886
"(35, 100]",0.000763


In [20]:
feature_df.groupby("client_has_ga4")["gsc_impressions"].mean()

,gsc_impressions
client_has_ga4,
False,37.314521
True,24.626067


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Signal tested: High impressions with a ranking position between 5 and 35 indicate good refresh candidates.

Verdict: SUPPORTED

Reason: The data shows that pages within better ranking positions generally achieve higher CTR, while pages with meaningful impressions have greater potential to gain additional traffic. This supports using impressions and average position together as a simple baseline signal for prioritizing content refreshes, although other factors such as seasonality and search intent should also be considered.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The results suggest that impressions and average search position are useful signals for prioritizing content refresh opportunities. However, they should be treated as decision-support signals rather than final decisions because factors such as seasonality, competition, technical SEO, and search intent are not captured by this analysis.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.